In [ ]:
import resource, traceback
def rss():
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20, 2)
def run(name, fn):
    print('START', name, 'rss GiB', rss(), flush=True)
    try:
        fn()
        print('DONE', name, 'rss GiB', rss(), flush=True)
    except Exception:
        print('FAILED', name, traceback.format_exc(), flush=True)


In [ ]:
import glob, torch
MODEL = glob.glob('/kaggle/input/**/Qwen3-Embedding-4B/config.json', recursive=True)[0].rsplit('/', 1)[0]
print('model dir', MODEL, 'cuda', torch.cuda.get_device_name(0))
def load():
    global tok, model
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = 'left'
    model = AutoModel.from_pretrained(MODEL, torch_dtype=torch.bfloat16, attn_implementation='sdpa').cuda().eval()
run('load model', load)


In [ ]:
def encode():
    batch = tok(['Instruct: test\nQuery: thuốc đau đầu'], padding=True, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model(**batch, use_cache=False).last_hidden_state
    print('hidden', tuple(out.shape), flush=True)
run('encode one query', encode)


In [ ]:
import numpy as np
def search():
    v = torch.from_numpy(np.load(glob.glob('/kaggle/input/**/vectors/en-part-000.npy', recursive=True)[0])).cuda().float()
    q = torch.randn(64, v.shape[1], device='cuda')
    s, r = torch.topk(q @ v.T, k=20, dim=1)
    print('topk', tuple(s.shape), 'vectors', tuple(v.shape), flush=True)
run('search one part', search)
print('probe complete', flush=True)
